# Demo 5 — L'inviluppo di una nota

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
t, nota, inv = al.tone_with_envelope(sr=SR)
rms = al.analysis.feature.rms(y=nota, frame_length=1024, hop_length=256)[0]
tempi = al.analysis.frames_to_time(np.arange(len(rms)), sr=SR, hop_length=256)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Nota sintetica con inviluppo di attacco e decadimento", "RMS a finestre"))
fig.add_trace(go.Scatter(x=t, y=nota, mode="lines", line=dict(width=1, color="#1769C2"),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=inv / np.sqrt(2), mode="lines", line=dict(width=1.2, color="#898989"),
                         name="inviluppo / √2 (RMS locale atteso)"), row=2, col=1)
fig.add_trace(go.Scatter(x=tempi, y=rms, mode="lines", line=dict(width=1.6, color="#005BFF"),
                         name="RMS a finestre"), row=2, col=1)
fig.update_xaxes(title_text="tempo [s]", row=2, col=1)
fig.update_layout(height=460, title="L'inviluppo di una nota")
fig.show()

display(al.safe_audio(nota, SR, normalize=False))
